In [ ]:
import math
import numpy as np
#from torch_geometric.data import Data, Dataset, DataLoader
#from torch_geometric.nn import GATv2Conv
#from torch_geometric.utils import to_undirected
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch_geometric.loader import DataLoader

import networkx as nx
from torch_geometric.utils import to_networkx
import matplotlib.pyplot as plt

## Creating the Model 

In [ ]:
# ------------ Model ------------

class GATShift(nn.Module):
    """
    GATv2-based node regression: predicts (dx, dy) per node.
    Uses edge_attr through GATv2Conv(edge_dim=...).
    """
    def __init__(self, in_dim, edge_dim, hidden=64, heads=4, layers=3, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList()
        self.activ = nn.ELU()
        self.dropout = nn.Dropout(dropout)

        # First
        self.layers.append(GATv2Conv(in_channels=in_dim, out_channels=hidden,
                                     heads=heads, edge_dim=edge_dim, dropout=dropout, concat=True))
        out_dim = hidden * heads

        # Middle
        for _ in range(layers - 2):
            self.layers.append(GATv2Conv(in_channels=out_dim, out_channels=hidden,
                                         heads=heads, edge_dim=edge_dim, dropout=dropout, concat=True))
            out_dim = hidden * heads

        # Last (set concat=False to keep output dim = hidden)
        self.layers.append(GATv2Conv(in_channels=out_dim, out_channels=hidden,
                                     heads=1, edge_dim=edge_dim, dropout=dropout, concat=False))

        # Regression head -> (dx, dy)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 2)
        )

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        for i, gat in enumerate(self.layers):
            x = gat(x, edge_index, edge_attr=edge_attr)
            if i < len(self.layers) - 1:
                x = self.activ(x)
                x = self.dropout(x)
        return self.head(x)  # [N,2]

In [ ]:
# ------------ Training loop ------------

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)

        # Mask: only synthetic_1 nodes (line_id == 1)
        line_ids = batch.x[:, 0]  # first column of features
        mask = (line_ids == 1)

        loss = F.mse_loss(pred[mask], batch.y[mask])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch)
            line_ids = batch.x[:, 0]
            mask = (line_ids == 1)
            loss = F.mse_loss(pred[mask], batch.y[mask])
            total_loss += loss.item()
    return total_loss / len(loader)


## Main Function

In [ ]:
if __name__ == "__main__":
    # Load your saved graphs
    graphs = torch.load(f'../data/final_dataset/graph/graphs_sequential.pt', weights_only=False)

    # Ensure we have a list (even if only one graph was saved)
    if not isinstance(graphs, list):
        graphs = [graphs]

    # Number of Subgraphs
    print(f"Number of Subgraphs: {len(graphs)}")

    # DataLoader (batch_size=1 for variable-size graphs)
    train_loader = DataLoader(graphs, batch_size=1, shuffle=True)

    # Use first graph for inferring dimensions
    sample = graphs[0]

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Model
    model = GATShift(
        in_dim=sample.x.size(1),
        edge_dim=sample.edge_attr.size(1) if sample.edge_attr is not None else 0,
        hidden=64,
        heads=4,
        layers=3,
        dropout=0.1
    ).to(device)

    # Optimizer
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

    # Train loop
    for epoch in range(1, 51):
        tr = train_epoch(model, train_loader, opt, device)
        va = eval_epoch(model, train_loader, device)  # Using same data for validation
        if epoch % 10 == 0:
            print(f"epoch {epoch:02d} | train MSE {tr:.5f} | val MSE {va:.5f}")

    # Inference on first graph
    model.eval()
    with torch.no_grad():
        graph = graphs[0].to(device)
        pred_shift = model(graph).cpu().numpy()  # [N, 2]

        # Mask: only synthetic_1 nodes
        line_ids = graph.x[:, 0].cpu().numpy()
        mask = line_ids == 1

        corrected_coords = graph.x[mask, 1:3].cpu().numpy() + pred_shift[mask]

        print("Predicted first 3 shifts (synthetic_1 nodes only):", pred_shift[mask][:3])
        print("Corrected first 3 coordinates:", corrected_coords[:3])


In [ ]:
# Inference on one polyline:
model.eval()
with torch.no_grad():
    graph = graph.to(device)
    pred_shift = model(graph).cpu().numpy()  # [N, 2]

    # Extract node coordinates for synthetic_1 nodes
    line_ids = graph.x[:, 0].cpu().numpy()
    mask = line_ids == 1  # Only synthetic_1 nodes

    coords_syn1 = graph.x[mask, 1:3].cpu().numpy()  # shape [num_nodes_syn1, 2]

    shifted_coords = coords_syn1 + pred_shift[mask]  # Apply predicted shift

    print("Predicted first 3 shifts:", pred_shift[mask][:3])
    print("Corrected first 3 coordinates:", shifted_coords[:3])


## Plotting the Result

In [ ]:
def plot_predicted_graph(graph, pred_shift, mask=None):
    """
    Plots a PyG graph with predicted shifts applied.

    graph      : torch_geometric.data.Data
    pred_shift : numpy array of shape [num_nodes, 2]
    mask       : boolean array to select specific nodes (optional)
    """
    # Convert PyG Data -> NetworkX
    G = to_networkx(graph, to_undirected=True)

    # Node positions: apply predicted shift
    coords = graph.x[:, 1:3].cpu().numpy()
    if mask is not None:
        coords = coords + pred_shift  # only masked nodes are shifted
    else:
        coords = coords + pred_shift

    pos = {i: (coords[i,0], coords[i,1]) for i in range(graph.num_nodes)}

    # Node colors by line_id
    color_map = ["#143642", "#EC9A29", "#A8201A"]
    line_ids = graph.x[:,0].cpu().numpy().astype(int)
    colors = [color_map[i] for i in line_ids]

    plt.figure(figsize=(8,6))
    nx.draw(G, pos,
            node_size=10,
            node_color=colors,
            edge_color="#B3B5B6A6",
            alpha=0.8)
    plt.title("Predicted Graph with Shifted Nodes")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.show()


In [ ]:
#Usage 
graph = graphs[0]
pred_shift = model(graph.to(device)).detach().cpu().numpy()

# Mask for synthetic_1 nodes (optional)
mask = graph.x[:,0].cpu().numpy() == 1
masked_shift = pred_shift.copy()
masked_shift[~mask] = 0  # only apply shift to masked nodes

plot_predicted_graph(graph, masked_shift, mask=mask)